In [10]:
import boto3
from datetime import datetime
import os
import io
import requests
import json
from app import db_utils, models
import gzip
import msgpack

def load_env_file(filepath=".env"):
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ[key] = value

load_env_file()
session = boto3.session.Session()

region_name = 'auto'
endpoint_url = os.environ['UPLOAD_ENDPOINT_URL']
aws_access_key_id = os.environ['AWS_ACCESS_KEY_ID']
aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY']
dev_url = os.environ['DEV_URL']

def write_to_r2(data, relative_path):
    client = session.client(
        's3',
        region_name='auto',
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key
    )

    json_bytes = io.BytesIO(json.dumps(data).encode('utf-8'))
    client.upload_fileobj(json_bytes, 'rshf', relative_path)
    return endpoint_url + '/' + relative_path


def read_from_r2(relative_path):
    response = requests.get(dev_url + '/' + relative_path)
    response.raise_for_status()
    return response.json()
    
def write_extension_data_to_r2():
    db = db_utils.SessionLocal()
    group_memberships = db.query(models.GroupMembership).all()
    db.close()
    data = dict()

    for obj in group_memberships:
        store_data = [
            obj.cf_handle,
            obj.user_group_rating
        ]
        if obj.group_id not in data:
            data[obj.group_id] = dict()
    
        data[obj.group_id][obj.user_id] = store_data
    
    res = {
        'timestamp': datetime.now().isoformat(),
        'data': data,
        'data_format': [
            'cf_handle', 'user_group_rating'
        ]
    }
    write_to_r2(res, 'extension_data.json')
    print(f"Finished writing {len(group_memberships)} entries to r2")
    read_url = dev_url + '/extension_data.json'
    return read_url

def read_extension_data_from_r2():
    return read_from_r2('extension_data.json')
    